# Решения: Проект 17 — Байесовский спам-классификатор

Решения упражнений из `17_Spam_Classifier.ipynb`. Данные и классификатор воспроизводятся здесь для самодостаточности.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

vocab = {
    'выигрыш':(0.30,0.02),'бесплатно':(0.35,0.03),'кредит':(0.25,0.02),
    'скидка':(0.28,0.05),'срочно':(0.22,0.04),'деньги':(0.30,0.06),
    'приз':(0.26,0.02),'встреча':(0.03,0.25),'проект':(0.04,0.28),
    'отчёт':(0.03,0.24),'коллега':(0.02,0.20),'договор':(0.05,0.22),
    'привет':(0.08,0.30),'спасибо':(0.06,0.28),'вопрос':(0.07,0.26),
}
words = list(vocab.keys())

def generate_email(is_spam, rng):
    idx = 0 if is_spam else 1
    tokens = [w for w in words if rng.random() < vocab[w][idx]]
    if not tokens:
        tokens = [rng.choice(words)]
    return ' '.join(tokens)

def make_dataset(n_spam, n_ham, seed=42):
    rng = np.random.default_rng(seed)
    data = [{'text': generate_email(True, rng), 'label': 'spam'} for _ in range(n_spam)]
    data += [{'text': generate_email(False, rng), 'label': 'ham'} for _ in range(n_ham)]
    return pd.DataFrame(data).sample(frac=1, random_state=seed).reset_index(drop=True)

df = make_dataset(400, 600)
print(f'Писем: {len(df)} | спам: {(df.label=="spam").sum()} | ham: {(df.label=="ham").sum()}')

### Реализация Naive Bayes (мультиномиальный, со сглаживанием Лапласа)
Обучение в лог-пространстве; `alpha` — параметр сглаживания.

In [ ]:
def train_nb(texts, labels, alpha=1.0):
    V = words
    model = {'logprior': {}, 'loglik': {}, 'alpha': alpha}
    N = len(labels)
    for cls in ['spam', 'ham']:
        docs = [t for t, l in zip(texts, labels) if l == cls]
        model['logprior'][cls] = np.log(len(docs) / N)
        counts = {w: 0 for w in V}
        total = 0
        for d in docs:
            for tok in d.split():
                counts[tok] += 1
                total += 1
        model['loglik'][cls] = {
            w: np.log((counts[w] + alpha) / (total + alpha * len(V))) for w in V}
    return model

def score_spam(model, text):
    logs = {}
    for cls in ['spam', 'ham']:
        s = model['logprior'][cls]
        for tok in text.split():
            if tok in model['loglik'][cls]:
                s += model['loglik'][cls][tok]
        logs[cls] = s
    m = max(logs.values())
    ps = np.exp(logs['spam'] - m)
    ph = np.exp(logs['ham'] - m)
    return ps / (ps + ph)  # P(spam|text)

def evaluate(model, texts, labels, thr=0.5):
    tp = fp = tn = fn = 0
    for t, l in zip(texts, labels):
        pred = 'spam' if score_spam(model, t) >= thr else 'ham'
        if l == 'spam' and pred == 'spam': tp += 1
        elif l == 'ham' and pred == 'spam': fp += 1
        elif l == 'ham' and pred == 'ham': tn += 1
        else: fn += 1
    prec = tp / (tp + fp) if tp + fp else 0
    rec = tp / (tp + fn) if tp + fn else 0
    f1 = 2*prec*rec/(prec+rec) if prec+rec else 0
    acc = (tp + tn) / (tp+fp+tn+fn)
    return dict(precision=prec, recall=rec, f1=f1, accuracy=acc,
                tp=tp, fp=fp, tn=tn, fn=fn)

# train/test split
split = int(0.7 * len(df))
train, test = df.iloc[:split], df.iloc[split:]
m = train_nb(train.text.tolist(), train.label.tolist(), alpha=1.0)
print('Базовое качество (alpha=1):', evaluate(m, test.text.tolist(), test.label.tolist()))

## Упражнение 1: Влияние сглаживания
Оценим F1 при разных `alpha` и построим график.

In [ ]:
alphas = [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10]
f1s = []
for a in alphas:
    ma = train_nb(train.text.tolist(), train.label.tolist(), alpha=a)
    r = evaluate(ma, test.text.tolist(), test.label.tolist())
    f1s.append(r['f1'])
    print(f'alpha={a:<5} F1={r["f1"]:.4f} precision={r["precision"]:.3f} recall={r["recall"]:.3f}')

plt.figure(figsize=(9, 5))
plt.semilogx(alphas, f1s, 'o-', color='steelblue')
plt.xlabel('alpha (сглаживание Лапласа, лог. шкала)')
plt.ylabel('F1')
plt.title('Зависимость F1 от параметра сглаживания')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print('Слишком малое alpha -> переобучение на редких словах;')
print('слишком большое -> сглаживает различия между классами.')

## Упражнение 2: Порог классификации и ROC-кривая
Меняем порог P(spam) и строим ROC. Если ложная пометка ham→spam особенно нежелательна, повышаем порог (снижаем FPR ценой recall).

In [ ]:
scores = np.array([score_spam(m, t) for t in test.text])
y = (test.label == 'spam').values.astype(int)

thrs = np.linspace(0, 1, 101)
tpr, fpr = [], []
for th in thrs:
    pred = scores >= th
    tp = np.sum(pred & (y == 1)); fn = np.sum(~pred & (y == 1))
    fp = np.sum(pred & (y == 0)); tn = np.sum(~pred & (y == 0))
    tpr.append(tp/(tp+fn) if tp+fn else 0)
    fpr.append(fp/(fp+tn) if fp+tn else 0)
auc = np.trapz(tpr[::-1], fpr[::-1])

plt.figure(figsize=(7, 7))
plt.plot(fpr, tpr, '-', color='coral', lw=2, label=f'ROC (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('FPR (ham помечен как spam)'); plt.ylabel('TPR (recall spam)')
plt.title('ROC-кривая спам-классификатора'); plt.legend()
plt.tight_layout(); plt.show()

# Порог для FPR <= 1%
for th in thrs:
    pred = scores >= th
    fp = np.sum(pred & (y == 0)); tn = np.sum(~pred & (y == 0))
    if fp/(fp+tn) <= 0.01:
        print(f'Порог для FPR<=1%: {th:.2f} (минимизируем ложные пометки ham)')
        break

## Упражнение 3: Несбалансированные классы (10% спама)
Сгенерируем 10% спама / 90% ham и покажем, почему accuracy обманчива.

In [ ]:
imb = make_dataset(100, 900, seed=1)
sp = int(0.7 * len(imb))
tr, te = imb.iloc[:sp], imb.iloc[sp:]
mi = train_nb(tr.text.tolist(), tr.label.tolist(), alpha=1.0)
r = evaluate(mi, te.text.tolist(), te.label.tolist())

baseline_acc = (te.label == 'ham').mean()  # всегда предсказываем ham
print(f'Несбалансированные данные (10% спам):')
print(f'  accuracy классификатора: {r["accuracy"]:.3f}')
print(f'  accuracy "всегда ham":    {baseline_acc:.3f}  <-- обманчиво высокая!')
print(f'  precision: {r["precision"]:.3f} | recall: {r["recall"]:.3f} | F1: {r["f1"]:.3f}')
print('\nВывод: при дисбалансе accuracy почти равна доле большинства класса.')
print('Нужно смотреть precision/recall/F1 по классу спама.')

## Упражнение 4: Сравнение с sklearn
Обучим `MultinomialNB` на тех же данных и сравним с нашей реализацией.

In [ ]:
try:
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score

    vec = CountVectorizer(vocabulary=words)
    Xtr = vec.transform(train.text); Xte = vec.transform(test.text)
    ytr = (train.label == 'spam').astype(int)
    yte = (test.label == 'spam').astype(int)
    clf = MultinomialNB(alpha=1.0).fit(Xtr, ytr)
    pred = clf.predict(Xte)
    pr, rc, f1, _ = precision_recall_fscore_support(yte, pred, average='binary')
    print('sklearn MultinomialNB:')
    print(f'  accuracy={accuracy_score(yte, pred):.3f} precision={pr:.3f} recall={rc:.3f} F1={f1:.3f}')
    own = evaluate(m, test.text.tolist(), test.label.tolist())
    print('Наша реализация:')
    print(f'  accuracy={own["accuracy"]:.3f} precision={own["precision"]:.3f} '
          f'recall={own["recall"]:.3f} F1={own["f1"]:.3f}')
    print('\nРезультаты близки: наша реализация повторяет мультиномиальный NB.')
except Exception as e:
    print('sklearn недоступен:', e)